# Solución P02: Parquet, contrato y reejecución

[Enunciado](../talleres/P02.md) · [Índice](README.md)

Ejecutar de arriba abajo con el kernel del entorno `.venv` preparado en la [guía WSL](../docs/instalacion-wsl.md). Usa las tres muestras incluidas; no descarga los archivos completos. Las salidas propias se regeneran al repetir la ejecución.

In [1]:
CODIGO = "P02"
from pathlib import Path
from types import SimpleNamespace
import sys, json
import pandas as pd
from IPython.display import display

# Funciona desde soluciones/, la raíz del repositorio o la carpeta base bigdata.
roots = [Path.cwd(), *Path.cwd().parents, Path.cwd() / 'big-data-postgraduate']
REPO = next((p for p in roots if (p / 'kit/pqrs_talleres.py').is_file()), None)
assert REPO is not None, 'Abrir Jupyter desde bigdata/big-data-postgraduate o soluciones/'
KIT = REPO / 'kit'
sys.path.insert(0, str(KIT))
import pqrs_talleres as lab
OUT = KIT / 'salidas/soluciones' / CODIGO
OUT.mkdir(parents=True, exist_ok=True)
a = SimpleNamespace(datos=None, completo=False, bloque=500, out=OUT, repeticiones=3)
print('Repositorio:', REPO, '\nSalidas:', OUT)


Repositorio: /opt/bigdata/big-data-postgraduate 
Salidas: /opt/bigdata/big-data-postgraduate/kit/salidas/soluciones/P02


## Tareas 1 y 2. Publicación trazable
El procedimiento verifica primero los hashes y conserva los CSV originales. La proyección tiene 16 columnas, frente a las 38 originales; `corte_archivo` y `fila_fuente` permiten volver al origen.

In [2]:
primera=lab.formatos(a)
lab.guardar(OUT/'primera_ejecucion.json',primera)
import pyarrow.parquet as pq
tabla=pq.read_table(OUT/'pqrs.parquet')
assert tabla.num_rows==3000 and tabla.num_columns==16
display(tabla.schema)
display(tabla.select(['corte_archivo','fila_fuente','OBJECTID']).slice(0,5).to_pandas())

[('2023_II', 1000, 752214), ('2024_I', 1000, 737025), ('2024_II', 1000, 745755)]
{'filas': 3000, 'columnas_producto': 16, 'columnas_original': 38, 'bytes_parquet': 65473, 'bytes_csv_proyeccion': 257499, 'particiones_archivos': 8, 'firma_contenido': '27186351908317914318628', 'bloque': 500, 'alcance': 'muestra', 'particion': ['anio', 'mes_num'], 'originales_modificados': False}


corte_archivo: string
fila_fuente: int64
OBJECTID: string
periodo: string
mes: string
afec_cod_depto: string
afec_cod_mpio: string
pet_cod_depto: string
pet_cod_mpio: string
ent_nombre: string
afec_edad: string
afec_genero: string
anio: int64
mes_num: int64
edad_num: double
clave_candidata: string

,corte_archivo,fila_fuente,OBJECTID
0,2023_II,2,1
1,2023_II,3,2
2,2023_II,4,3
3,2023_II,5,4
4,2023_II,6,5


## Tarea 3. Archivo simple y particiones
La partición procede de `periodo` y `mes`, no del semestre del nombre de archivo. Conciliamos ambos productos y mostramos el costo de almacenamiento.

In [3]:
import duckdb
with duckdb.connect() as con:
    filas=con.execute('SELECT count(*) FROM read_parquet(?)',[str(OUT/'pqrs_particionado/**/*.parquet')]).fetchone()[0]
assert filas==tabla.num_rows
partes=list((OUT/'pqrs_particionado').rglob('*.parquet'))
assert len(partes)==8
display(pd.DataFrame({'archivo':[str(p.relative_to(OUT)) for p in partes], 'bytes':[p.stat().st_size for p in partes]}))
print('Bytes simple:',(OUT/'pqrs.parquet').stat().st_size,'; particionado:',sum(p.stat().st_size for p in partes))

,archivo,bytes
0,pqrs_particionado/anio=2023/mes_num=7/data_0.p...,14641
1,pqrs_particionado/anio=2024/mes_num=2/data_0.p...,2588
2,pqrs_particionado/anio=2024/mes_num=3/data_0.p...,3049
3,pqrs_particionado/anio=2024/mes_num=5/data_0.p...,6038
4,pqrs_particionado/anio=2024/mes_num=1/data_0.p...,2729
5,pqrs_particionado/anio=2024/mes_num=4/data_0.p...,3690
6,pqrs_particionado/anio=2024/mes_num=6/data_0.p...,12916
7,pqrs_particionado/anio=2024/mes_num=7/data_0.p...,14448


Bytes simple: 65473 ; particionado: 60099


## Tarea 4. Reejecución
Repetir reemplaza los productos, no agrega filas. La firma comprueba estabilidad del control seleccionado; no demuestra igualdad de todas las propiedades semánticas.

In [4]:
segunda=lab.formatos(a)
for key in ['filas','firma_contenido','columnas_producto']:
    assert primera[key]==segunda[key], key
lab.guardar(OUT/'segunda_ejecucion.json',segunda)
display(pd.DataFrame([primera,segunda],index=['primera','segunda']))

[('2023_II', 1000, 752214), ('2024_I', 1000, 737025), ('2024_II', 1000, 745755)]
{'filas': 3000, 'columnas_producto': 16, 'columnas_original': 38, 'bytes_parquet': 65473, 'bytes_csv_proyeccion': 257499, 'particiones_archivos': 8, 'firma_contenido': '27186351908317914318628', 'bloque': 500, 'alcance': 'muestra', 'particion': ['anio', 'mes_num'], 'originales_modificados': False}


,filas,columnas_producto,columnas_original,bytes_parquet,bytes_csv_proyeccion,particiones_archivos,firma_contenido,bloque,alcance,particion,originales_modificados
primera,3000,16,38,65473,257499,8,27186351908317914318628,500,muestra,"[anio, mes_num]",False
segunda,3000,16,38,65473,257499,8,27186351908317914318628,500,muestra,"[anio, mes_num]",False


## Tarea 5. Fallo de contrato en memoria
Retiramos una columna de una copia. El contrato debe impedir la publicación; el archivo original no se modifica.

In [5]:
f,p=lab.seleccion(a)[0]
original=next(lab.leer(f,p,100))
defectuoso=original.drop(columns=['mes'])
try:
    lab.normalizar(defectuoso,f,0)
except ValueError as error:
    incidencia={'columna_retirada':'mes','error':str(error),'accion':'detener publicación'}
else:
    raise AssertionError('El contrato no detectó la columna ausente')
lab.guardar(OUT/'incidencia_contrato.json',incidencia)
restaurado=lab.normalizar(original,f,0)
assert restaurado.num_rows==100
display(incidencia)
lab.guardar(OUT/'configuracion_solucion.json',{'bloque':a.bloque,'completo':False})

{'columna_retirada': 'mes',
 'error': 'Contrato de nombres incumplido: detener antes de publicar',
 'accion': 'detener publicación'}

**Decisión:** publicar solo tras verificar origen, contrato y conciliación. Restaurar la columna elimina este fallo concreto, pero no reemplaza los controles semánticos. Particionar puede producir archivos pequeños y empeorar el rendimiento; no implica una mejora automática. La entrega incluye controles de ambas ejecuciones, configuración e incidencia.